Text-to-Speech with the Azure Speech SDK

The **Speech Synthesis** (text-to-speech / TTS) API turns text into natural-sounding spoken audio using neural voices. This notebook starts from a minimal "speak one sentence" call and builds toward the fuller capabilities of the service:

- Configuring credentials and choosing a neural voice
- Confirming success and handling errors on the result object
- Listing the hundreds of available voices
- Writing audio to a file and picking an output format
- **SSML** for fine control over pace, pitch, pauses, and speaking style
- Word-boundary events for building captions

In [1]:
!pip install azure-cognitiveservices-speech


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup — imports, credentials, and configuration

`azure.cognitiveservices.speech` (aliased `speechsdk`) is the client library. `load_dotenv()` reads a local `.env` file into environment variables. `SpeechConfig` carries your credentials and voice/format choices; `AudioOutputConfig(use_default_speaker=True)` sends synthesized audio to the machine's default speaker.

In [2]:
import os
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk
from pathlib import Path

In [3]:
load_dotenv(Path.cwd().parent /".env")

True

In [4]:
key = os.getenv("AZURE_SPEECH_KEY")
region = os.getenv("AZURE_SPEECH_REGION")

In [6]:
speech_config = speechsdk.SpeechConfig(subscription=key, region=region)
audio_config = speechsdk.audio.AudioOutputConfig(use_default_speaker=True)

## Exploring the configuration object

The next two cells use `help()` and `dir()` purely to explore what `SpeechConfig` exposes — its methods (`set_speech_synthesis_output_format`, `set_profanity`, ...) and properties (`speech_synthesis_voice_name`, `region`, ...). Introspecting the object like this is a good habit when learning any new SDK.

In [7]:
help(speech_config)

Help on SpeechConfig in module azure.cognitiveservices.speech object:

class SpeechConfig(builtins.object)
 |  SpeechConfig(
 |      subscription: Optional[str] = None,
 |      region: Optional[str] = None,
 |      endpoint: Optional[str] = None,
 |      host: Optional[str] = None,
 |      auth_token: Optional[str] = None,
 |      speech_recognition_language: Optional[str] = None,
 |      token_credential: Optional['TokenCredential'] = None,
 |      key_credential: Optional['AzureKeyCredential'] = None
 |  )
 |
 |  Class that defines configurations for speech recognition and speech synthesis.
 |
 |  The configuration can be initialized in different ways:
 |
 |  - from subscription: pass a subscription key and a region
 |  - from endpoint: pass an endpoint. Subscription key, AzureKeyCredential, or authorization token are optional.
 |  - from host: pass a host address. Subscription key or authorization token are optional.
 |  - from authorization token: pass an authorization token and a 

In [8]:
[i for i in dir(speech_config) if not i.startswith("_")]

['authorization_token',
 'enable_audio_logging',
 'enable_dictation',
 'endpoint_id',
 'get_property',
 'get_property_by_name',
 'key_credential',
 'output_format',
 'region',
 'request_word_level_timestamps',
 'set_profanity',
 'set_properties',
 'set_properties_by_name',
 'set_property',
 'set_property_by_name',
 'set_proxy',
 'set_service_property',
 'set_speech_synthesis_output_format',
 'speech_recognition_language',
 'speech_synthesis_language',
 'speech_synthesis_output_format_string',
 'speech_synthesis_voice_name',
 'subscription_key',
 'token_credential']

## Choosing a voice and synthesizing

Setting `speech_synthesis_voice_name` selects one of the neural voices (here a Bulgarian voice, `bg-BG-BorislavNeural`). The `SpeechSynthesizer` binds the config (voice + credentials) to the audio output (speaker). `speak_text_async(...).get()` runs synthesis on a background thread and blocks on `.get()` for the result; `speak_text(...)` is the simpler fully-blocking form. Both return a `SpeechSynthesisResult`.

In [32]:
speech_config.speech_synthesis_voice_name = "en-US-Jane:DragonHDLatestNeural"

In [33]:
speech_client = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)

In [34]:
speech_client.speak_text_async("Hello, my name is Jane. To be or not to be, that is the question. My kingdom for a horse").get()

In [29]:
speech_client.speak_text("Здравейте, това е тест на SDK за реч на Azure Cognitive Services.")

## Checking the result and handling errors

Synthesis can fail — a wrong key, an unavailable region, or a mistyped voice name. The returned `SpeechSynthesisResult` carries a `reason`; on failure, `cancellation_details` explains what went wrong. Real code should always check it rather than assume success.

In [24]:
# Every speak_*_async call returns a SpeechSynthesisResult. Inspect it to
# confirm success or to diagnose a failure.
result = speech_client.speak_text_async(
    "asdasd, Hello World. Здравейте това е тест на SDK за реч на Azure Cognitive Services."
).get()  # .get() blocks until synthesis finishes, then returns the result

# reason tells you what happened. On success it is SynthesizingAudioCompleted.
if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    # audio_data holds the raw synthesized bytes (also useful for saving/streaming).
    print(f"Synthesis complete: {len(result.audio_data)} bytes of audio")
elif result.reason == speechsdk.ResultReason.Canceled:
    # cancellation_details explains WHY the service rejected the request.
    details = result.cancellation_details
    print(f"Canceled: {details.reason}")
    print(f"Error details: {details.error_details}")

Synthesis complete: 254446 bytes of audio


## Listing the available voices

The service offers hundreds of neural voices across languages, genders, and speaking styles. `get_voices_async()` retrieves the full catalog so you can choose the right voice programmatically instead of hardcoding a name.

In [25]:
# get_voices_async() asks the service for every voice available in your region.
voices_result = speech_client.get_voices_async().get()

if voices_result.reason == speechsdk.ResultReason.VoicesListRetrieved:
    voices = voices_result.voices
    print(f"Total voices available: {len(voices)}\n")

    # Show the US English neural voices and any expressive styles they support.
    for v in voices:
        if v.locale == "en-US":
            styles = ", ".join(v.style_list) if v.style_list else "(no styles)"
            # short_name is the value you assign to speech_synthesis_voice_name.
            print(f"{v.short_name:32} {v.gender.name:7} styles: {styles}")
else:
    print(f"Could not retrieve voices: {voices_result.reason}")

Total voices available: 769

en-US-Ava:DragonHDLatestNeural   Female  styles: 
en-US-Andrew:DragonHDLatestNeural Male    styles: 
en-US-Adam:DragonHDLatestNeural  Male    styles: 
en-US-Alloy:DragonHDLatestNeural Male    styles: 
en-US-Aria:DragonHDLatestNeural  Female  styles: 
en-US-Bree:DragonHDLatestNeural  Female  styles: 
en-US-Brian:DragonHDLatestNeural Male    styles: 
en-US-Davis:DragonHDLatestNeural Male    styles: 
en-US-Emma:DragonHDLatestNeural  Female  styles: 
en-US-Emma2:DragonHDLatestNeural Female  styles: 
en-US-Jane:DragonHDLatestNeural  Female  styles: 
en-US-Jenny:DragonHDLatestNeural Female  styles: 
en-US-Nova:DragonHDLatestNeural  Female  styles: 
en-US-Phoebe:DragonHDLatestNeural Female  styles: 
en-US-Serena:DragonHDLatestNeural Female  styles: 
en-US-Steffan:DragonHDLatestNeural Male    styles: 
en-US-Andrew:DragonHDOmniLatestNeural Male    styles: 
en-US-Caleb:DragonHDOmniLatestNeural Male    styles: 
en-US-Dana:DragonHDOmniLatestNeural Female  styles: 
en-U

## Synthesizing to a file and choosing an output format

Point `AudioOutputConfig` at a filename to save audio to disk instead of playing it through the speaker, and call `set_speech_synthesis_output_format` to control the codec and quality (for example, a compact MP3). The output format must be set **before** the synthesizer is created.

In [35]:
# A fresh config so we don't disturb the speaker-based one used above.
file_speech_config = speechsdk.SpeechConfig(subscription=key, region=region)
file_speech_config.speech_synthesis_voice_name = "en-US-JennyNeural"

# Pick a compact MP3 output format (set BEFORE creating the synthesizer).
file_speech_config.set_speech_synthesis_output_format(
    speechsdk.SpeechSynthesisOutputFormat.Audio16Khz32KBitRateMonoMp3
)

os.makedirs("Data", exist_ok=True)

# Giving AudioOutputConfig a filename routes audio to disk instead of the speaker.
file_config = speechsdk.audio.AudioOutputConfig(filename="Data/tts_output.mp3")

file_synth = speechsdk.SpeechSynthesizer(
    speech_config=file_speech_config, audio_config=file_config
)
result = file_synth.speak_text_async(
    "This narration is being written to an audio file on disk."
).get()
print(f"Wrote {len(result.audio_data)} bytes to Data/tts_output.mp3")

Wrote 16128 bytes to Data/tts_output.mp3


## SSML — fine-grained control over delivery

**Speech Synthesis Markup Language** wraps your text in XML that controls pace (`prosody rate`), pitch, pauses (`break`), volume, and pronunciation. Use `speak_ssml_async` instead of `speak_text_async`, and note that the voice is chosen *inside* the markup. Ready-made SSML samples also live in `Data/ssml.xml` and `Data/ssml2.xml`.

> **Note on `<emphasis>`.** The standard `<emphasis>` tag is a leftover from the older (concatenative) voices and is **effectively ignored by Azure neural voices** — the service accepts the markup and returns success, but you won't hear any difference. Neural models generate prosody holistically and don't honor a word-level emphasis hint. To actually stress a word, reach for `<prosody>` (raise `volume`, slow `rate`, lift `pitch`) or a `mstts:express-as` speaking style. The cell below uses `<prosody>` on "approved" for exactly this reason.

In [41]:
# SSML is an XML document. The voice is named inside it, so it overrides the
# voice set on the config. Here: an emphasized word, a pause, and slowed/raised delivery.
#
# NOTE: <emphasis> is ignored by neural voices, so we stress "approved" with
# <prosody> instead — louder, slightly slower, and higher-pitched. That combination
# is what neural voices actually respond to.
# ssml = """
# <speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="en-US">
#   <voice name="en-US-JennyNeural">
#     Your transaction was
#     <prosody volume="+50.00%" rate="-80%" pitch="-4st">approved</prosody>.
#     <break time="200ms"/>
#     <prosody rate="+50%" pitch="+4st">
#       Please retain this confirmation number for your records.
#     </prosody>
#   </voice>
# </speak>
# """

# # speak_ssml_async parses the markup rather than treating the input as plain text.
# result = speech_client.speak_ssml_async(ssml).get()
# print(result.reason)

# You can also read SSML straight from a file:
with open("Data/ssml.xml", "r", encoding="utf-8") as f:
    speech_client.speak_ssml_async(f.read()).get()

## Speaking styles and multi-voice dialogue

Neural voices support expressive **styles** (`customerservice`, `cheerful`, `sad`, ...) via the `mstts:express-as` element, and you can switch voices within a single document to script a conversation — useful for IVR prompts, training audio, and simulated support calls. Note the extra `xmlns:mstts` namespace declaration required for the style element.

In [42]:
# Two voices in one document = a scripted dialogue. The mstts namespace enables
# express-as, which applies a speaking style to the enclosed text.
dialogue_ssml = """
<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis"
       xmlns:mstts="http://www.w3.org/2001/mstts" xml:lang="en-US">
  <voice name="en-US-JennyNeural">
    <mstts:express-as style="customerservice">
      Thank you for calling support. How can I help you today?
    </mstts:express-as>
  </voice>
  <voice name="en-US-GuyNeural">
    I have a question about a duplicate charge on my account.
  </voice>
</speak>
"""
result = speech_client.speak_ssml_async(dialogue_ssml).get()
print(result.reason)

ResultReason.SynthesizingAudioCompleted
